# Tasks List

In [ ]:
## Tasks List

# Make sure that the Player's bet does not exceed their available chips
# The dealer will always Hit until the Dealer's value meets or exceeds 17

# Refactor with Hand() class
# Refactor ace handling
# Refactor with CoPilot recommendations
# Add a tie situation

# Setup classes

In [1]:
## Blackjack Game
## Kyaw Soe
## 11th Oct 2024
## v1

import random

# Define card parameters
suits = ('Hearts', 'Diamonds', 'Spades', 'Clubs')
ranks = ('Two', 'Three', 'Four', 'Five', 'Six', 'Seven', 'Eight', 'Nine', 'Ten', 'Jack', 'Queen', 'King', 'Ace')
values = {'Two':2, 'Three':3, 'Four':4, 'Five':5, 'Six':6, 'Seven':7, 'Eight':8, 
            'Nine':9, 'Ten':10, 'Jack':10, 'Queen':10, 'King':10, 'Ace':11}

## Setup game ##

# Setup card class
class Card:
   
    def __init__(self,suit,rank):
    # def __init__(self,suit,rank,face_up):
        self.suit = suit
        self.rank = rank
        # self.face_up = face_up
        # not sure why this was added
        self.value = values[rank]
                
    def __str__(self):
        return '  - ' + self.rank + ' of ' + self.suit + ' value ' + str(self.value)

    
# Setup deck class     
class Deck:
    
    def __init__(self):
        # Create an empty deck
        self.all_cards = []
        
        for suit in suits:
            for rank in ranks:
                self.all_cards.append(Card(suit,rank))

    def shuffle(self):
        # Randomly shuffle cards
        random.shuffle(self.all_cards)
        
    def deal_one(self):
        # Note we remove one card from the list of all_cards
        return self.all_cards.pop() 
    
    def __str__(self):
        return self.all_cards
    
    def __len__(self):
        return len(self.all_cards)
    
    
# Setup player class
class Player:
    
    def __init__(self,name,money):
        #Player starts with no cards
        self.all_cards = []
        self.value = 0
        self.name = name
        self.money = money
                
    def add_cards(self,new_cards):
        if type(new_cards) == type([]):
            self.all_cards.extend(new_cards)
        else:
            self.all_cards.append(new_cards)
    
    def list_cards(self):
        for item in self.all_cards:
            print(item, sep='\n')
    
    def cards_value(self):
        # for item in self.all_cards:
        #     value += item.value
        #     #print(item.value)
        value = 0
        ace_count = 0
        extra_value = 0

        for item in self.all_cards:
            # Check for ace
            if item.rank == 'Ace':
                ace_count += 1
                if value + 11 > 21:
                    value += 1
                else:
                    value += 1
                    extra_value = 10
            # Add values if not ace
            else:
                value += item.value
            #print(item.rank)
            #print(item.value)

        # If aces included, calculate a non-bust value
        if value + extra_value < 22:
            value = value + extra_value
        # No aces value
        else:
            value = value

        # Return total value of cards
        # print(f'Ace count: {ace_count}')
        # print(f'Total value: {value}')
        
        # Return total value of cards
        self.value = value
        return self.value
        
    def __str__(self):
        return f'Player {self.name} has £{self.money} and has {len(self.all_cards)} cards'
    
    def __len__(self):
        return len(self.all_cards)
    
    


# Define game functions

In [2]:
# Setup game end conditions
def check_result(player_hand,dealer_hand):
    
    if player_hand > 21:
        print('\nGame over! Player busts, dealer wins.')
        return 'Player busts!'

    if dealer_hand > 21:
        print('\nGame over! Dealer busts, player wins.')
        return 'Dealer busts!'   
    
    if player_hand == 21:
        print('\nGame over! Player wins.')
        return 'Player wins!'
    
    if dealer_hand > player_hand:
        print('\nGame over, dealer wins.')
        return 'Dealer wins!'

    return 'Continues'


# Prompt player
def hit_stay(player_name):
    turn = ''

    while turn not in ['h','s']:
        try:
            turn = input(f'{player_name} please choose Hit(h) or Stay(s): ')
            turn = turn.lower()
        except ValueError:
            print('Please enter h(hit) or s(stay).')
         
    return turn


# Create player and place bet ##
def createPlayer():
    # New players and bankroll
    name = input('What is your name? ')
    bankroll = 100
    human_player = Player(name, bankroll)
    computer_player = Player('Dealer', 10000)
    print(human_player)

    return human_player, computer_player


# Place bet
def placeBet():
    while True:
        request_bet = input('How much would you like to bet? ')
        try:
            player_bet = int(request_bet)

            if player_bet > human_player.money:
                print('You do no have sufficient funds! Please enter a smaller bet.\n')
            else:
                player_bet = int(request_bet)
                break

        except ValueError:
            print('Please enter an integer number.')

    print(f"\nYou have bet £{player_bet}\n")
    return player_bet


## Deal initial cards ##
def dealCards(play_deck):
    # Shuffle a new deck if cards are low (or new deck)
    if len(play_deck) < 10:
        print('Suffling a new deck.\n')
        play_deck = Deck()
        play_deck.shuffle()

    # Deal two cards to human player
    draw_card = play_deck.deal_one()
    human_player.add_cards(draw_card)

    draw_card = play_deck.deal_one()
    human_player.add_cards(draw_card)

    print('Player draws:')
    human_player.list_cards()

    # Deal two cards to computer player
    draw_card = play_deck.deal_one()
    computer_player.add_cards(draw_card)
    print('\nDealer draws:')
    print(computer_player.all_cards[0])

    draw_card = play_deck.deal_one()
    computer_player.add_cards(draw_card)
    print('  - A second card face down\n')
    
    return play_deck, human_player, computer_player


def hitStay():
    # Player hits until wins/loses
    result = ''
    turn = ''

    while turn == '':

        # Player already won
        # Check if player's first two cards are 21
        if human_player.cards_value() == 21:
            print('Player has 21!')
            turn = 's'
            #player has 21, does not matter what dealer has
            result = check_result(21,0) 
        else:
            # Players hit/stay choice
            turn = hit_stay(human_player.name)

            # Player hits
            if turn == 'h':
                draw_card = play_deck.deal_one()
                human_player.add_cards(draw_card)

                player_hand = human_player.cards_value()
                dealer_hand = computer_player.cards_value()

                print(f'\n{human_player.name} draws:\n{draw_card}')
                print(f'\n{human_player.name} holds {len(human_player)} cards:')
                human_player.list_cards()

                print(f'\n{human_player.name} hand is {player_hand}')

                # Check results
                # Dealer hand set to 0 before dealer plays
                result = check_result(player_hand,0)
                ##print(result) #debugging

                if result == 'Continues':
                    turn = ''
                else:
                    print(result)
                    break

            # Player stays
            elif turn == 's':

                # Dealer shows hand
                print(f'\nDealer holds {len(computer_player)} cards:')
                computer_player.list_cards()

                # Calculate hands
                player_hand = human_player.cards_value()
                dealer_hand = computer_player.cards_value()

                #Check results
                result = check_result(player_hand,dealer_hand)
                ##print(result) #debugging

                if result == 'Continues':
                    pass
                else:
                    ##print(result) #debugging
                    break
    
    return result

            
# Dealer draws until game ends
def dealerDraws(result):
    while result not in ['Player wins!','Player busts!','Dealer wins!','Dealer busts!']:

            draw_card = play_deck.deal_one()
            computer_player.add_cards(draw_card)

            # Calculate hands
            player_hand = human_player.cards_value()
            dealer_hand = computer_player.cards_value()

            print(f'\n{computer_player.name} draws:\n{draw_card}')
            print(f'\nDealer holds {len(computer_player)} cards:')
            computer_player.list_cards()

            print(f'\n{human_player.name} hand is {player_hand}')
            print(f'{computer_player.name} hand is {dealer_hand}')

            result = check_result(player_hand,dealer_hand)
            
    return result
    

# Calculate bets
def calculateBets(player_bet, result):
    if result in ['Player wins!', 'Dealer busts!']:
        human_player.money = human_player.money + 2*player_bet 
        computer_player.money = computer_player.money - 2*player_bet
        print(f'{human_player.name} wins £{2*player_bet}')
        print(f'You now have £{human_player.money}')
        print(f'Dealer now has £{computer_player.money}')

    else:
        human_player.money = human_player.money - player_bet 
        computer_player.money = computer_player.money + player_bet
        print(f'{human_player.name} loses bet of £{player_bet}')
        print(f'{human_player.name}, you now have £{human_player.money}')
        print(f'Dealer now has £{computer_player.money}')


# Play a round

In [17]:
# Main

# Setup an inital playing deck
print('Suffling a new deck.\n')
play_deck = Deck()
play_deck.shuffle()

# Create players
[human_player, computer_player] = createPlayer()


while human_player.money > 0:

    # Place bet
    player_bet = placeBet()

    # Deal cards
    [play_deck, human_player, computer_player] = dealCards(play_deck)

    # Player's turn
    result = hitStay()
    # print(f'Debug: {result}') #debugging
    
    # Check results
    if result not in ['Player wins!','Player busts!','Dealer wins!','Dealer busts!']:
        # Dealer draws until game ends
        result = dealerDraws(result)
    else:
        pass
    
    # print(f'Debug: {result}') #debugging
    
    # Calculate bets
    calculateBets(player_bet, result)

    # Empty hands
    computer_player.all_cards = []
    human_player.all_cards = []
    
    # Keep playing until player hits quit
    keepPlaying = ''
    while keepPlaying not in ['q','c']:
        try:
            keepPlaying = input('\nQuit(q) or Continue playing(c): ')
            keepPlaying = keepPlaying.lower()
        except ValueError:
            print('Please enter q(quit) or c(continue).')
            
    if keepPlaying == 'q':
        break
    else:
        pass
    
if human_player.money <= 0:
    print(f'{human_player.name}, you have run out of money and cannot play another round.')
else:
    print('Thank you for playing.')


Suffling a new deck.

What is your name? Kyaw
Player Kyaw has £100 and has 0 cards
How much would you like to bet? 50

You have bet £50

Player draws:
  - Ace of Diamonds value 11
  - Ten of Hearts value 10

Dealer draws:
  - King of Clubs value 10
  - A second card face down

Player has 21!

Game over! Player wins.
Kyaw wins £100
You now have £200
Dealer now has £9900

Quit(q) or Continue playing(c): d

Quit(q) or Continue playing(c): d

Quit(q) or Continue playing(c): c
How much would you like to bet? 200

You have bet £200

Player draws:
  - King of Spades value 10
  - Four of Hearts value 4

Dealer draws:
  - Seven of Spades value 7
  - A second card face down

Kyaw please choose Hit(h) or Stay(s): s

Dealer holds 2 cards:
  - Seven of Spades value 7
  - Six of Hearts value 6

Dealer draws:
  - Jack of Hearts value 10

Dealer holds 3 cards:
  - Seven of Spades value 7
  - Six of Hearts value 6
  - Jack of Hearts value 10

Kyaw hand is 14
Dealer hand is 23

Game over! Dealer busts, 